In [ ]:
# Colab setup - run this first. Installs m3 plus the tutorial plotting extras.
# (Colab already ships PyTorch, which m3 uses as its engine.)
%pip install -q "git+https://github.com/PYangLab/M3.git" scanpy umap-learn

# Donor-level disease prediction

Leave-one-batch-out donor prediction on the built-in Liu subsample.
Uses the SAME validated parameters as the full-data run — only the data is
subsampled, so the held-out subset is smaller and the numbers may differ slightly
from the full-data run.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
import m3

OUT = "./tutorial_out_demo/02_patient_prediction"
os.makedirs(OUT, exist_ok=True)

In [ ]:
data = m3.datasets.liu_demo()
print(data)

## Build with held-out B3

In [ ]:
model = m3.M3(
    data,
    condition_keys=["cond_group", "Age_interval"],
    target_condition="cond_group",
    celltype_key="mergedcelltype",
    batch_key="batch",
    donor_key="sample_id",
    held_out=["B3"],
    embedding_dim=30,
)
print("reference vocab:", model.contract["reference_vocab"]["cond_group"])

In [ ]:
#To save time, users can set max_epochs to 100 for test.
model.train(
    max_epochs=500,
    donor_predictor={
        "glr": 3e-3, "n_epochs": 120, "adv_max": 10, "adv_warmup": 7,
        "n_disc": 21, "patient_w": 10,
    },
)
print("capabilities:", model.capabilities)

## Predict held-out donors

In [ ]:
preds = model.predict_donors()
print("query donors:", len(preds))
print(preds.to_string(index=False))
preds.to_csv(os.path.join(OUT, "predictions.csv"), index=False)

In [ ]:
donor_truth = (data.obs[["sample_id", "cond_group"]].astype(str)
               .drop_duplicates().set_index("sample_id")["cond_group"].to_dict())
preds["true_label"] = preds["donor"].map(donor_truth)
pos = "Severe"
y_true = (preds["true_label"] == pos).astype(int).to_numpy()
y_score = preds[f"prob_{pos}"].to_numpy()
acc = float((preds["predicted_label"] == preds["true_label"]).mean())
auc = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else float("nan")
print(f"held-out accuracy = {acc:.3f}   ROC-AUC = {auc:.3f}")

## ROC curve

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
if len(np.unique(y_true)) > 1:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    ax.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="Held-out donor ROC")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "donor_roc.png"), dpi=130, bbox_inches="tight")
plt.show()

## Patient-level embedding (UMAP)

In [ ]:
import umap
import h5py
import anndata as ad

demb = model.donor_embedding()
info = model.predict_donors(include_reference=True).set_index("donor").reindex(demb.index)
info["true_label"] = [donor_truth.get(d) for d in demb.index]
info["set"] = np.where(info["is_reference"], "reference", "query")
info["correct"] = np.where(
    info["is_reference"].to_numpy(), "reference",
    np.where(info["predicted_label"].to_numpy() == info["true_label"].to_numpy(),
             "correct", "wrong"))

X = demb.drop(columns=["is_reference"]).to_numpy()
xy = umap.UMAP(n_neighbors=min(15, len(X) - 1), random_state=0).fit_transform(X)
plot_df = info.assign(x=xy[:, 0], y=xy[:, 1])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key in zip(axes, ["set", "true_label", "correct"]):
    for val, sub in plot_df.groupby(key):
        ax.scatter(sub["x"], sub["y"], s=60, alpha=0.85, label=str(val))
    ax.set(title=f"Patient embedding — {key}", xticks=[], yticks=[])
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "patient_embedding_umap.png"), dpi=130, bbox_inches="tight")
plt.show()

# save patient-level
np.save(os.path.join(OUT, "patient_embedding.npy"), X)
with h5py.File(os.path.join(OUT, "patient_embedding.h5"), "w") as f:
    f.create_dataset("data", data=X)
    f.create_dataset("donor_name", data=np.asarray(list(demb.index), dtype="S"))
    f.create_dataset("label", data=np.asarray(info["true_label"].astype(str).to_numpy(), dtype="S"))
    f.create_dataset("is_reference", data=info["is_reference"].to_numpy())
ad.AnnData(X=X, obs=info[["set", "true_label", "correct"]].astype(str)).write_h5ad(
    os.path.join(OUT, "patient_embedding.h5ad"))
print("saved patient embedding -> .npy / .h5 (data+donor_name+label) / .h5ad in", OUT)